# RL's Razor: Why Online RL Forgets Less

This notebook reproduces the ParityMNIST experiment from the paper "RL's Razor: Why Online Reinforcement Learning Forgets Less" ([arXiv:2509.04259v1](https://arxiv.org/html/2509.04259v1)).

## Key Questions We're Investigating:

1. **Does RL forget less than SFT?** We'll compare retention on FashionMNIST after fine-tuning on ParityMNIST.
2. **Does KL shift predict forgetting?** We'll plot forgetting vs KL(base||ft) measured on the new task.
3. **Is on-policy sampling the key?** We'll compare different training methods (SFT, REINFORCE, oracle SFT).

## Experimental Setup:

- **Base model**: MLP pretrained jointly on FashionMNIST + ParityMNIST
- **New task**: ParityMNIST only (fine-tune to improve parity accuracy)
- **Old task**: FashionMNIST (measure retention/forgetting)
- **Methods**: SFT (fixed labels), SFT (oracle labels), REINFORCE (on-policy)

## Step 1: Setup and Imports

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np

from rl_razor_paritymnist.config import ExperimentConfig, TrainConfig
from rl_razor_paritymnist.data import get_dataloaders
from rl_razor_paritymnist.models import MLP
from rl_razor_paritymnist.training import (
    pretrain_joint,
    finetune_sft_fixed,
    finetune_sft_oracle,
    finetune_reinforce
)
from rl_razor_paritymnist.evaluation import fashion_accuracy, parity_success, kl_base_to_ft
from rl_razor_paritymnist.utils import set_seed

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## Step 2: Configure Experiment

Adjust these parameters to control the experiment.

In [ ]:
# Create configuration
config = ExperimentConfig.default()

# Update device
config.train.device = DEVICE

# Optional: adjust training steps
config.train.pretrain_steps = 800
config.train.finetune_steps = 400
config.train.lr = 1e-3
config.train.seed = 42

print("Experiment Configuration:")
print(f"  Pretrain steps: {config.train.pretrain_steps}")
print(f"  Finetune steps: {config.train.finetune_steps}")
print(f"  Learning rate: {config.train.lr}")
print(f"  Batch size: {config.train.batch_size}")
print(f"  Seed: {config.train.seed}")

## Step 3: Load Data

We'll create dataloaders for:
- **ParityMNIST**: The new task we'll fine-tune on
- **FashionMNIST**: The old task we want to retain

In [ ]:
print("Loading datasets...")
loaders = get_dataloaders(config)

fashion_train_loader = loaders['fashion_train']
parity_train_loader = loaders['parity_train']
fashion_test_loader = loaders['fashion_test']
parity_test_loader = loaders['parity_test']

print(f"✓ FashionMNIST train batches: {len(fashion_train_loader)}")
print(f"✓ ParityMNIST train batches: {len(parity_train_loader)}")
print(f"✓ FashionMNIST test samples: {len(fashion_test_loader.dataset)}")
print(f"✓ ParityMNIST test samples: {len(parity_test_loader.dataset)}")

## Step 4: Pretrain Base Model

We pretrain an MLP jointly on both tasks. This gives us a base model that has some capability on both tasks.

In [ ]:
# Set seed for reproducibility
set_seed(config.train.seed)

# Create base model
base_model = MLP(
    hidden1=config.model.hidden1,
    hidden2=config.model.hidden2,
    out_dim=config.model.out_dim
).to(DEVICE)

print(f"Base model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

# Pretrain
print("\nPretraining base model...")
pretrain_joint(
    base_model,
    parity_train_loader,
    fashion_train_loader,
    config.train,
    device=DEVICE
)

print("\nPretraining complete!")

## Step 5: Evaluate Base Model

Let's check the base model's performance on both tasks before fine-tuning.

In [ ]:
base_fashion_acc = fashion_accuracy(base_model, fashion_test_loader, device=DEVICE)
base_parity_succ = parity_success(base_model, parity_test_loader, device=DEVICE)

print("Base Model Performance:")
print(f"  FashionMNIST accuracy: {base_fashion_acc:.3f}")
print(f"  ParityMNIST success:   {base_parity_succ:.3f}")
print(f"\nNote: We'll measure 'forgetting' as the drop in FashionMNIST accuracy after fine-tuning.")

## Step 6: Fine-tune with Different Methods

We'll fine-tune the base model using four different approaches:

1. **SFT (fixed 0/1)**: Always use label 0 for even, 1 for odd (arbitrary choice, may cause large KL)
2. **SFT (random)**: Pick random but fixed even/odd labels
3. **REINFORCE**: On-policy RL with binary reward
4. **SFT (oracle)**: Use minimum-KL labels (sampling from base model restricted to correct answers)

### 6.1: SFT with Fixed 0/1 Mapping

In [ ]:
print("Training: SFT with fixed 0/1 mapping...")
set_seed(config.train.seed)

model_sft_01 = MLP(
    hidden1=config.model.hidden1,
    hidden2=config.model.hidden2,
    out_dim=config.model.out_dim
).to(DEVICE)
model_sft_01.load_state_dict(base_model.state_dict())

finetune_sft_fixed(
    model_sft_01,
    parity_train_loader,
    config.train,
    device=DEVICE,
    mapping="01"
)

print("✓ SFT (fixed 0/1) complete")

### 6.2: SFT with Random Mapping

In [ ]:
print("Training: SFT with random mapping...")
set_seed(config.train.seed + 1)

model_sft_rand = MLP(
    hidden1=config.model.hidden1,
    hidden2=config.model.hidden2,
    out_dim=config.model.out_dim
).to(DEVICE)
model_sft_rand.load_state_dict(base_model.state_dict())

finetune_sft_fixed(
    model_sft_rand,
    parity_train_loader,
    config.train,
    device=DEVICE,
    mapping="random"
)

print("✓ SFT (random) complete")

### 6.3: REINFORCE (On-Policy RL)

In [ ]:
print("Training: REINFORCE (on-policy RL)...")
set_seed(config.train.seed + 2)

model_reinforce = MLP(
    hidden1=config.model.hidden1,
    hidden2=config.model.hidden2,
    out_dim=config.model.out_dim
).to(DEVICE)
model_reinforce.load_state_dict(base_model.state_dict())

finetune_reinforce(
    model_reinforce,
    parity_train_loader,
    config.train,
    device=DEVICE
)

print("✓ REINFORCE complete")

### 6.4: SFT with Oracle Labels

In [ ]:
print("Training: SFT with oracle (min-KL) labels...")
set_seed(config.train.seed + 3)

model_sft_oracle = MLP(
    hidden1=config.model.hidden1,
    hidden2=config.model.hidden2,
    out_dim=config.model.out_dim
).to(DEVICE)
model_sft_oracle.load_state_dict(base_model.state_dict())

finetune_sft_oracle(
    model_sft_oracle,
    base_model,
    parity_train_loader,
    config.train,
    device=DEVICE
)

print("✓ SFT (oracle) complete")

## Step 7: Evaluate All Models

Now we evaluate each fine-tuned model on:
- **New task performance**: ParityMNIST success rate
- **Old task retention**: FashionMNIST accuracy
- **KL shift**: KL(base || fine-tuned) on ParityMNIST

In [ ]:
def evaluate_model(name, model):
    """Evaluate a model and return metrics."""
    parity_succ = parity_success(model, parity_test_loader, device=DEVICE)
    fashion_acc = fashion_accuracy(model, fashion_test_loader, device=DEVICE)
    kl = kl_base_to_ft(base_model, model, parity_test_loader, device=DEVICE)
    forgetting = base_fashion_acc - fashion_acc
    
    print(f"{name:>18s} | parity={parity_succ:.3f} fashion={fashion_acc:.3f} KL={kl:.4f} forgetting={forgetting:.3f}")
    
    return {
        'name': name,
        'parity_success': parity_succ,
        'fashion_accuracy': fashion_acc,
        'kl': kl,
        'forgetting': forgetting
    }

print("\nEvaluating all models...\n")
results = []

results.append(evaluate_model("SFT (fixed 0/1)", model_sft_01))
results.append(evaluate_model("SFT (random)", model_sft_rand))
results.append(evaluate_model("REINFORCE", model_reinforce))
results.append(evaluate_model("SFT (oracle)", model_sft_oracle))

print("\n✓ Evaluation complete")

## Step 8: Visualize Results

### 8.1: Forgetting vs KL Shift

The paper's key claim: forgetting is predicted by KL shift on the new task.

In [ ]:
plt.figure(figsize=(10, 6))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
markers = ['o', 's', '^', 'd']

for i, result in enumerate(results):
    plt.scatter(
        result['kl'],
        result['forgetting'],
        s=200,
        c=colors[i],
        marker=markers[i],
        alpha=0.7,
        edgecolors='black',
        linewidth=1.5,
        label=f"{result['name']} (parity={result['parity_success']:.2f})"
    )

plt.xlabel("KL(base || fine-tuned) on ParityMNIST", fontsize=12)
plt.ylabel("Forgetting on FashionMNIST\n(base_acc - ft_acc)", fontsize=12)
plt.title("Forgetting vs KL Shift: RL's Razor Reproduction", fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('forgetting_vs_kl.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved as: forgetting_vs_kl.png")

### 8.2: Pareto Frontier (New Task vs Old Task)

In [ ]:
plt.figure(figsize=(10, 6))

# Plot base model
plt.scatter(
    base_parity_succ,
    base_fashion_acc,
    s=250,
    c='gold',
    marker='*',
    alpha=0.9,
    edgecolors='black',
    linewidth=2,
    label='Base Model',
    zorder=10
)

# Plot fine-tuned models
for i, result in enumerate(results):
    plt.scatter(
        result['parity_success'],
        result['fashion_accuracy'],
        s=200,
        c=colors[i],
        marker=markers[i],
        alpha=0.7,
        edgecolors='black',
        linewidth=1.5,
        label=f"{result['name']} (KL={result['kl']:.3f})"
    )

plt.xlabel("ParityMNIST Success Rate (New Task)", fontsize=12)
plt.ylabel("FashionMNIST Accuracy (Old Task)", fontsize=12)
plt.title("Pareto Frontier: New Task Performance vs Old Task Retention", fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('pareto_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved as: pareto_frontier.png")

### 8.3: Summary Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df[['name', 'parity_success', 'fashion_accuracy', 'kl', 'forgetting']]
df.columns = ['Method', 'Parity Success', 'Fashion Accuracy', 'KL Shift', 'Forgetting']

# Add base model row
base_row = pd.DataFrame([{
    'Method': 'Base Model',
    'Parity Success': base_parity_succ,
    'Fashion Accuracy': base_fashion_acc,
    'KL Shift': 0.0,
    'Forgetting': 0.0
}])

df = pd.concat([base_row, df], ignore_index=True)

print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(df.to_string(index=False))
print("="*80)

# Save to CSV
df.to_csv('results.csv', index=False)
print("\nResults saved to: results.csv")

## Step 9: Key Findings

Let's summarize what we learned:

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Find best method by retention
best_retention = max(results, key=lambda x: x['fashion_accuracy'])
print(f"\n1. Best retention: {best_retention['name']}")
print(f"   - Fashion accuracy: {best_retention['fashion_accuracy']:.3f}")
print(f"   - Parity success: {best_retention['parity_success']:.3f}")
print(f"   - KL shift: {best_retention['kl']:.4f}")

# Find method with lowest KL
lowest_kl = min(results, key=lambda x: x['kl'])
print(f"\n2. Lowest KL shift: {lowest_kl['name']}")
print(f"   - KL shift: {lowest_kl['kl']:.4f}")
print(f"   - Forgetting: {lowest_kl['forgetting']:.3f}")

# Compare REINFORCE vs SFT
reinforce_result = [r for r in results if 'REINFORCE' in r['name']][0]
sft_results = [r for r in results if 'SFT' in r['name'] and 'oracle' not in r['name']]
avg_sft_kl = np.mean([r['kl'] for r in sft_results])
avg_sft_forgetting = np.mean([r['forgetting'] for r in sft_results])

print(f"\n3. REINFORCE vs Average SFT (non-oracle):")
print(f"   - REINFORCE KL: {reinforce_result['kl']:.4f} | Avg SFT KL: {avg_sft_kl:.4f}")
print(f"   - REINFORCE forgetting: {reinforce_result['forgetting']:.3f} | Avg SFT forgetting: {avg_sft_forgetting:.3f}")
print(f"   - RL forgets {(avg_sft_forgetting - reinforce_result['forgetting']):.3f} less than SFT on average")

# Oracle insight
oracle_result = [r for r in results if 'oracle' in r['name']][0]
print(f"\n4. Oracle SFT (minimum-KL labels):")
print(f"   - Demonstrates that KL is the key factor")
print(f"   - Oracle forgetting: {oracle_result['forgetting']:.3f}")
print(f"   - Oracle KL: {oracle_result['kl']:.4f}")
print(f"   - Shows that SFT *can* forget less if labels are chosen to minimize KL")

print("\n" + "="*80)
print("CONCLUSION: On-policy RL naturally finds solutions with lower KL shift,")
print("which leads to less forgetting. The 'RL's Razor' principle is confirmed!")
print("="*80)

## Optional: Save Models

Uncomment to save trained models for later analysis.

In [ ]:
# import os
# os.makedirs('checkpoints', exist_ok=True)

# torch.save(base_model.state_dict(), 'checkpoints/base_model.pt')
# torch.save(model_sft_01.state_dict(), 'checkpoints/sft_01.pt')
# torch.save(model_sft_rand.state_dict(), 'checkpoints/sft_random.pt')
# torch.save(model_reinforce.state_dict(), 'checkpoints/reinforce.pt')
# torch.save(model_sft_oracle.state_dict(), 'checkpoints/sft_oracle.pt')

# print("Models saved to checkpoints/")